In [ ]:
import sys
import argparse
import os
sys.path.append(os.path.expanduser("~/websites/mapedia"))
from modules import DBHandler, DBUpdater
import pandas as pd
import numpy as np
from modules.db_handler.DBConfig import INTER_CITY_LEARNING_SOURCE, INTRA_CITY_LEARNING_SOURCE, EMPTY_SOURCE
INTRA_CITY_LEARNING_PRESET_CONFIDENCE = 0.8

In [16]:
db_handler = DBHandler()
db_handler.connect_to_db()
db_updater = DBUpdater(db_handler)

In [3]:
metadata_path  = os.path.join('./data/imputed_data/', f"jakarta_imputedBy_jakarta.parquet")
metadata = pd.read_parquet(metadata_path)
metadata.head()

,idx,source,target,mapd_id,pgr_id,osm_id,oneway,road_type,nlanes,width,...,imputed_avg_speed_weekend_08-12,avg_speed_weekend_12-16,pred_avg_speed_weekend_12-16,imputed_avg_speed_weekend_12-16,avg_speed_weekend_16-20,pred_avg_speed_weekend_16-20,imputed_avg_speed_weekend_16-20,avg_speed_weekend_20-24,pred_avg_speed_weekend_20-24,imputed_avg_speed_weekend_20-24
0,0,52860,7874,282664,98,4705045,1.0,2.0,4.0,10.0,...,8.897243,8.955077,41.415527,8.955077,11.846818,48.392166,11.846818,5.271908,46.888874,5.271908
1,1,30445,87959,282663,212,4705043,1.0,2.0,1.0,4.0,...,7.279488,7.610105,22.801670,7.610105,11.480000,24.459898,11.480000,9.215501,24.294548,9.215501
2,2,1359,102290,282665,213,4705046,1.0,2.0,2.0,6.0,...,8.752268,9.169477,28.757809,9.169477,12.175380,31.591429,12.175380,15.253763,31.416620,15.253763
3,3,47817,58347,282665,38783,4705046,1.0,2.0,2.0,6.0,...,8.752268,9.169477,42.240459,9.169477,12.175380,46.035675,12.175380,15.253763,48.690834,15.253763
4,4,7874,92444,282664,47381,4705045,1.0,2.0,4.0,10.0,...,8.897243,8.955077,35.345345,8.955077,11.846818,39.710392,11.846818,5.271908,39.743553,5.271908


In [4]:
metadata.columns

Index(['idx', 'source', 'target', 'mapd_id', 'pgr_id', 'osm_id', 'oneway',
       'road_type', 'nlanes', 'width', 'length', 'geometry', 'max_speed',
       'min_speed', 'nlanes_cls', 'highway_id', 'pred_road_type',
       'pred_nlanes_cls', 'pred_oneway', 'pred_width', 'pred_max_speed',
       'pred_min_speed', 'imputed_road_type', 'imputed_nlanes_cls',
       'imputed_oneway', 'imputed_width', 'imputed_max_speed',
       'imputed_min_speed', 'avg_speed_weekday_00-04',
       'pred_avg_speed_weekday_00-04', 'imputed_avg_speed_weekday_00-04',
       'avg_speed_weekday_04-08', 'pred_avg_speed_weekday_04-08',
       'imputed_avg_speed_weekday_04-08', 'avg_speed_weekday_08-12',
       'pred_avg_speed_weekday_08-12', 'imputed_avg_speed_weekday_08-12',
       'avg_speed_weekday_12-16', 'pred_avg_speed_weekday_12-16',
       'imputed_avg_speed_weekday_12-16', 'avg_speed_weekday_16-20',
       'pred_avg_speed_weekday_16-20', 'imputed_avg_speed_weekday_16-20',
       'avg_speed_weekday_20-24', 

In [5]:
pred_metadata = metadata[['mapd_id', 'osm_id',
        'pred_road_type', 'pred_nlanes_cls',
       'pred_oneway', 'pred_width', 'pred_max_speed', 'pred_min_speed']].copy()
pred_metadata.rename(columns={
    'pred_road_type': 'road_type', 
    'pred_nlanes_cls': 'nlanes',
    'pred_oneway': 'oneway', 
    'pred_width': 'width', 
    'pred_max_speed': 'max_speed', 
    'pred_min_speed': 'min_speed'
}, inplace=True)

In [6]:
for c in ['road_type', 'nlanes', 'oneway', 'width', 'max_speed', 'min_speed']:
    pred_metadata[f'{c}_source'] = INTRA_CITY_LEARNING_SOURCE
    pred_metadata[f'{c}_conf'] = INTRA_CITY_LEARNING_PRESET_CONFIDENCE

In [7]:
speed_metadata = metadata[['mapd_id', 'osm_id',
       'pred_avg_speed_weekday_00-04',
       'pred_avg_speed_weekday_04-08',
       'pred_avg_speed_weekday_08-12',
       'pred_avg_speed_weekday_12-16',
       'pred_avg_speed_weekday_16-20',
       'pred_avg_speed_weekday_20-24',
       'pred_avg_speed_weekend_00-04',
       'pred_avg_speed_weekend_04-08',
       'pred_avg_speed_weekend_08-12',
       'pred_avg_speed_weekend_12-16',
       'pred_avg_speed_weekend_16-20',
       'pred_avg_speed_weekend_20-24']].copy()



In [8]:
# Period label → (start_hour, end_hour)
PERIOD_HOURS = {
    "00-04": range(0, 4),
    "04-08": range(4, 8),
    "08-12": range(8, 12),
    "12-16": range(12, 16),
    "16-20": range(16, 20),
    "20-24": range(20, 24),
}

# weekday=0 means Monday in pandas; 0-4 = weekday, 5-6 = weekend
WEEKDAY_DAYS = list(range(5))   # 0-4
WEEKEND_DAYS = list(range(5, 7)) # 5-6

# 1. One period-block: 6 periods × their hour counts = 24 values
#    Each period value repeated for its hours
period_cols_weekday = [f"pred_avg_speed_weekday_{p}" for p in PERIOD_HOURS]
period_cols_weekend = [f"pred_avg_speed_weekend_{p}" for p in PERIOD_HOURS]
period_lengths      = [len(h) for h in PERIOD_HOURS.values()]  # [4,4,4,4,4,4]

def build_day_vector(row, cols):
    """24 values: each period value repeated for its hour count"""
    return np.repeat([row[c] for c in cols], period_lengths)  # (24,)

def build_week_vector(row):
    """168 values: 5× weekday-day + 2× weekend-day"""
    day = build_day_vector(row, period_cols_weekday)  # (24,)
    end = build_day_vector(row, period_cols_weekend)  # (24,)
    return np.concatenate([np.tile(day, 5), np.tile(end, 2)])  # (168,)

def build_speed_array(row):
    """672 values: week vector repeated 4× for seasons"""
    return np.tile(build_week_vector(row), 4)  # (672,)

# Apply once per road
speed_matrix = np.stack(speed_metadata.apply(build_speed_array, axis=1))  # (N, 672)

In [9]:
pred_metadata["avg_speed"] = list(speed_matrix)
pred_metadata["avg_speed_source"] = [np.full(672, INTRA_CITY_LEARNING_SOURCE, dtype=object)] * len(pred_metadata)
pred_metadata["avg_speed_conf"]   = list(np.full((len(pred_metadata), 672), INTRA_CITY_LEARNING_PRESET_CONFIDENCE, dtype=np.float32))
dynamic_pred_metadata = pred_metadata[['mapd_id', 'osm_id', 'avg_speed', 'avg_speed_source', 'avg_speed_conf']].copy()
static_pred_metadata = pred_metadata.drop(columns=['avg_speed', 'avg_speed_source', 'avg_speed_conf'])

In [10]:
dynamic_pred_metadata = dynamic_pred_metadata.set_index("mapd_id").rename_axis(None)
static_pred_metadata = static_pred_metadata.set_index("mapd_id").rename_axis(None)

In [11]:
dynamic_pred_metadata.shape

(512991, 4)

In [81]:
id = dynamic_pred_metadata.iloc[0].name
new_source = dynamic_pred_metadata.iloc[0].avg_speed_source.astype(float)
new_conf = dynamic_pred_metadata.iloc[0].avg_speed_conf.astype(float)
new_val = dynamic_pred_metadata.iloc[0].avg_speed

In [82]:
old_val = np.array(db_updater.road_attributes.loc[id].avg_speed, dtype=float)
old_source = np.array(db_updater.road_attributes.loc[id].avg_speed_source, dtype=float)
old_conf = np.array(db_updater.road_attributes.loc[id].avg_speed_conf, dtype=float)

In [ ]:
new_source = dynamic_pred_metadata.iloc[0].avg_speed_source.astype(float)
new_conf = dynamic_pred_metadata.iloc[0].avg_speed_conf.astype(float)
new_val = dynamic_pred_metadata.iloc[0].avg_speed

In [ ]:
db_updater.road_attributes.avg_speed_source.apply(lambda x: np.array(x, dtype=float))

In [ ]:
new_source = dynamic_pred_metadata.iloc[0].avg_speed_source.astype(float)
new_conf = dynamic_pred_metadata.iloc[0].avg_speed_conf.astype(float)

In [ ]:
db_updater.road_attributes.

In [ ]:
EMPTY_SOURCE = 9999 # don't use for any other source


array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,

In [84]:
empty_old_val = np.isnan(old_val)
empty_old_source = np.isnan(old_source)
empty_new_source = np.isnan(new_source)

In [ ]:
old_source = np.nan_to_num(old_source, nan=EMPTY_SOURCE)
new_source = np.nan_to_num(new_source, nan=EMPTY_SOURCE)
higher_priority_source = new_source < old_source

In [ ]:
old_nonempty_source = (old_source != EMPTY_SOURCE)
new_nonempty_source = (new_source != EMPTY_SOURCE)
old_conf = np.nan_to_num(old_conf, nan=0.0)
new_conf = np.nan_to_num(new_conf, nan=0.0)
higher_conf = (old_nonempty_source & new_nonempty_source & (new_conf >= old_conf))

In [70]:
old_is_empty | higher_priority_source | higher_conf

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,

In [ ]:
old_source = db_updater.road_attributes.loc[id]

osm_id                                                        4705045
oneway                                                            1.0
road_type                                                         2.0
width                                                            10.0
nlanes                                                            4.0
max_speed                                                         NaN
min_speed                                                         NaN
avg_speed           [None, None, None, None, None, None, None, Non...
oneway_source                                                     0.0
road_type_source                                                  0.0
width_source                                                      0.0
nlanes_source                                                     0.0
max_speed_source                                                  NaN
min_speed_source                                                  NaN
avg_speed_source    

In [101]:
db_updater.road_attributes.avg_speed = db_updater.road_attributes.avg_speed.apply(lambda x: np.array(x, dtype=float))
db_updater.road_attributes.avg_speed_conf = db_updater.road_attributes.avg_speed_conf.apply(lambda x: np.array(x, dtype=float))
db_updater.road_attributes.avg_speed_source = db_updater.road_attributes.avg_speed_source.apply(lambda x: np.array(x, dtype=float))
dynamic_pred_metadata.avg_speed_source = dynamic_pred_metadata.avg_speed_source.apply(lambda x: x.astype(float))
dynamic_pred_metadata.avg_speed_conf = dynamic_pred_metadata.avg_speed_conf.apply(lambda x: x.astype(float))

In [102]:
road_attributes = db_updater.road_attributes
def update_temporal_attributes_new(row):
    id = row.name
    old_row = road_attributes.loc[id]

    for col in ['avg_speed']:
        old_val    = old_row[col]
        old_source = old_row[f"{col}_source"]
        old_conf   = old_row[f"{col}_conf"]

        new_val    = row[col]                   # already (672,)
        new_source = row[f"{col}_source"]
        new_conf   = row[f"{col}_conf"]

        empty_old_val = np.isnan(old_val)
        better_source = (new_source < old_source)
        same_source_better_conf = (new_source == old_source) & (new_conf >= old_conf)
        mask = empty_old_val | better_source | same_source_better_conf

        old_val[mask]    = new_val[mask]
        old_source[mask] = new_source[mask]
        old_conf[mask]   = new_conf[mask]

    road_attributes.loc[id] = old_row

tqdm.pandas(desc="Updating temporal road attributes")
dynamic_pred_metadata.progress_apply(update_temporal_attributes_new, axis=1)

Updating temporal road attributes:  16%|█▌        | 83292/512991 [03:06<16:03, 446.16it/s]


KeyboardInterrupt: 

In [ ]:
import numpy as np
import pandas as pd

print('1. Align the DataFrames by ID so the matrix rows match perfectly')
shared_ids = dynamic_pred_metadata.index
road_attr_subset = road_attributes.loc[shared_ids]

print('2. Extract entire columns into 2D NumPy arrays')
# Each resulting array will have a shape of (num_rows, 672)
old_val = np.stack(road_attr_subset['avg_speed'].tolist())
old_source = np.stack(road_attr_subset['avg_speed_source'].tolist())
old_conf = np.stack(road_attr_subset['avg_speed_conf'].tolist())

new_val = np.stack(dynamic_pred_metadata['avg_speed'].tolist())
new_source = np.stack(dynamic_pred_metadata['avg_speed_source'].tolist())
new_conf = np.stack(dynamic_pred_metadata['avg_speed_conf'].tolist())

print('3. Compute the boolean mask across the entire 2D grid instantly')
empty_old_val = np.isnan(old_val)
better_source = new_source < old_source
same_source_better_conf = (new_source == old_source) & (new_conf >= old_conf)

# Combine conditions into a single master mask
mask = empty_old_val | better_source | same_source_better_conf

print('4. Apply changes to the arrays in-place where the mask is True')
old_val[mask] = new_val[mask]
old_source[mask] = new_source[mask]
old_conf[mask] = new_conf[mask]

# print('5. Convert modified matrices back into Python lists to update the DataFrame')
# (We wrap them back in individual lists so Pandas can store them in cell objects)
# Convert the 2D matrix back into a 1D list of individual arrays/lists
# road_attributes.loc[shared_ids, 'avg_speed'] = [row for row in old_val]
# road_attributes.loc[shared_ids, 'avg_speed_source'] = [row for row in old_source]
# road_attributes.loc[shared_ids, 'avg_speed_conf'] = [row for row in old_conf]

1. Align the DataFrames by ID so the matrix rows match perfectly
2. Extract entire columns into 2D NumPy arrays
3. Compute the boolean mask across the entire 2D grid instantly
4. Apply changes to the arrays in-place where the mask is True
5. Convert modified matrices back into Python lists to update the DataFrame


ValueError: Must have equal len keys and value when setting with an ndarray

In [144]:
id_list = shared_ids.tolist()
db_handler = DBHandler()
db_handler.connect_to_db()
db_updater = DBUpdater(db_handler)

In [ ]:
db_handler.update_dynamic_attributes(old_val, old_source, old_conf, id_list)

Connecting to Database using psycopg2
Set Up Chunking
Create High-Performance UNLOGGED Staging Table


In [136]:
old_val[0][0] = np.nan

In [138]:
str(old_val[0].tolist())

'[nan, 39.29716873168945, 39.29716873168945, 39.29716873168945, 38.240020751953125, 38.240020751953125, 38.240020751953125, 38.240020751953125, 35.73617172241211, 35.73617172241211, 35.73617172241211, 35.73617172241211, 40.179359436035156, 40.179359436035156, 40.179359436035156, 40.179359436035156, 48.85234451293945, 48.85234451293945, 48.85234451293945, 48.85234451293945, 49.44483947753906, 49.44483947753906, 49.44483947753906, 49.44483947753906, 39.29716873168945, 39.29716873168945, 39.29716873168945, 39.29716873168945, 38.240020751953125, 38.240020751953125, 38.240020751953125, 38.240020751953125, 35.73617172241211, 35.73617172241211, 35.73617172241211, 35.73617172241211, 40.179359436035156, 40.179359436035156, 40.179359436035156, 40.179359436035156, 48.85234451293945, 48.85234451293945, 48.85234451293945, 48.85234451293945, 49.44483947753906, 49.44483947753906, 49.44483947753906, 49.44483947753906, 39.29716873168945, 39.29716873168945, 39.29716873168945, 39.29716873168945, 38.24002

In [139]:
str(old_val[0].tolist()).replace("[", "{").replace("]", "}")

'{nan, 39.29716873168945, 39.29716873168945, 39.29716873168945, 38.240020751953125, 38.240020751953125, 38.240020751953125, 38.240020751953125, 35.73617172241211, 35.73617172241211, 35.73617172241211, 35.73617172241211, 40.179359436035156, 40.179359436035156, 40.179359436035156, 40.179359436035156, 48.85234451293945, 48.85234451293945, 48.85234451293945, 48.85234451293945, 49.44483947753906, 49.44483947753906, 49.44483947753906, 49.44483947753906, 39.29716873168945, 39.29716873168945, 39.29716873168945, 39.29716873168945, 38.240020751953125, 38.240020751953125, 38.240020751953125, 38.240020751953125, 35.73617172241211, 35.73617172241211, 35.73617172241211, 35.73617172241211, 40.179359436035156, 40.179359436035156, 40.179359436035156, 40.179359436035156, 48.85234451293945, 48.85234451293945, 48.85234451293945, 48.85234451293945, 49.44483947753906, 49.44483947753906, 49.44483947753906, 49.44483947753906, 39.29716873168945, 39.29716873168945, 39.29716873168945, 39.29716873168945, 38.24002

In [111]:
old_source.astype(int)

/tmp/ipykernel_858428/97585962.py:1: RuntimeWarning: invalid value encountered in cast
  old_source.astype(int)


array([[4, 4, 4, ..., 4, 4, 4],
       [4, 4, 4, ..., 4, 4, 4],
       [4, 4, 4, ..., 4, 4, 4],
       ...,
       [4, 4, 4, ..., 4, 4, 4],
       [4, 4, 4, ..., 4, 4, 4],
       [4, 4, 4, ..., 4, 4, 4]])

In [109]:
# Convert rows to plain Python tuples or a tuple of arrays to block 2D unpacking
road_attributes.loc[shared_ids, 'avg_speed'] = [tuple(row) for row in old_val]
road_attributes.loc[shared_ids, 'avg_speed_source'] = [tuple(row) for row in old_source]
road_attributes.loc[shared_ids, 'avg_speed_conf'] = [tuple(row) for row in old_conf]

ValueError: Must have equal len keys and value when setting with an ndarray

In [107]:
len(shared_ids)

512991

In [99]:
db_handler = DBHandler()
db_handler.connect_to_db()
db_updater = DBUpdater(db_handler)
db_updater.update_database_new(temporal_attr=dynamic_pred_metadata)

Updating temporal road attributes:   0%|          | 1/512991 [00:00<15:24:50,  9.24it/s]


TypeError: ufunc 'isnan' not supported for the input types, and the inputs could not be safely coerced to any supported types according to the casting rule ''safe''

In [100]:
db_updater.road_attributes.iloc[0].avg_speed

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,

In [ ]:
db_updater.update_database_new(static_attr=static_pred_metadata, temporal_attr=dynamic_pred_metadata)

In [14]:
dynamic_pred_metadata

,osm_id,avg_speed,avg_speed_source,avg_speed_conf
282664,4705045,"[39.29716873168945, 39.29716873168945, 39.2971...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, ..."
282663,4705043,"[22.1138973236084, 22.1138973236084, 22.113897...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, ..."
282665,4705046,"[27.154027938842773, 27.154027938842773, 27.15...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, ..."
282665,4705046,"[38.18931579589844, 38.18931579589844, 38.1893...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, ..."
282664,4705045,"[33.255165100097656, 33.255165100097656, 33.25...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, ..."
...,...,...,...,...
212245832,1323780385,"[36.438865661621094, 36.438865661621094, 36.43...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, ..."
211625167,1318113065,"[23.50935935974121, 23.50935935974121, 23.5093...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, ..."
211603264,1317986765,"[34.763431549072266, 34.763431549072266, 34.76...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, ..."
211553340,1317609752,"[23.481294631958008, 23.481294631958008, 23.48...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, ..."


In [38]:
import geopandas as gpd
edges_path  = os.path.join('./data/raw_data', f"jakarta_edges.parquet")
edges = gpd.read_parquet(edges_path)

In [39]:
edges

,source,target,pgr_id,osm_id,oneway,road_type,nlanes,width,length,geometry,max_speed,min_speed
0,52860,7874,98,4705045,1.0,2.0,4.0,10.000000,56.769815,"LINESTRING (106.84 -6.1673, 106.84 -6.1677)",NaN,NaN
1,30445,87959,212,4705043,1.0,2.0,1.0,4.000000,293.579384,"LINESTRING (106.84 -6.1678, 106.84 -6.1679, 10...",NaN,NaN
2,1359,102290,213,4705046,1.0,2.0,2.0,6.000000,12.779379,"LINESTRING (106.84 -6.1681, 106.84 -6.1682)",NaN,NaN
3,47817,58347,38783,4705046,1.0,2.0,2.0,6.000000,76.590609,"LINESTRING (106.84 -6.1669, 106.84 -6.167, 106...",NaN,NaN
4,7874,92444,47381,4705045,1.0,2.0,4.0,10.000000,28.160970,"LINESTRING (106.84 -6.1677, 106.84 -6.168)",NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
512986,554572,3126680,4265252,1323780385,1.0,3.0,1.0,7.670237,5.210311,"LINESTRING (106.84 -6.1815, 106.84 -6.1815)",NaN,NaN
512987,562231,3126704,4265281,1318113065,0.0,1.0,1.0,2.319501,28.838543,"LINESTRING (106.76 -6.1815, 106.76 -6.1814, 10...",NaN,NaN
512988,1705639,3126735,4265318,1317986765,0.0,0.0,0.0,3.734968,27.817565,"LINESTRING (106.76 -6.1698, 106.76 -6.1697)",NaN,NaN
512989,1590678,3126756,4265341,1317609752,0.0,0.0,1.0,3.889288,30.114436,"LINESTRING (106.69 -6.1917, 106.69 -6.1917, 10...",NaN,NaN
